# 🎬 AI YouTube Automation — Video Engine

**AI-powered YouTube video & Shorts generation engine**

This notebook runs the full pipeline: from a single motivational quote to a fully-rendered YouTube video — **no API tokens required!**

| Step | Module | What it does |
|------|--------|--------------|
| 1 | Story Generation | Creates a unique story using Ollama LLM (Gemma3) |
| 2 | SEO Metadata | Generates title, description, and hashtags |
| 3 | Image Prompt | Summarises the story into a visual prompt |
| 4 | Image Generation | Creates images locally using **Stable Diffusion XL** (GPU) |
| 5 | Audio Generation | Converts story to speech (Kokoro TTS) |
| 6 | Transcription | Generates subtitles from audio (Whisper) |
| 7 | Subtitle Processing | Converts SRT → JSON for MoviePy |
| 8 | Video Assembly | Renders landscape video + YouTube Shorts |
| 9 | YouTube Upload | OAuth2 upload with scheduling |

> ⚡ **GPU Required**: Select **T4 GPU** from `Runtime > Change runtime type` for image generation and faster Whisper transcription.

---

## 📦 Step 1 — Clone Repository & Install Dependencies

This cell clones the project from GitHub and installs all required Python packages including PyTorch, Diffusers, and Whisper.

In [ ]:
# Clone the repository
!git clone https://github.com/tamilarasu18/ai-youtube-automation.git
%cd ai-youtube-automation

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

print("\n✅ Installation complete!")

## 🔧 Step 2 — Install System Dependencies

Install `ffmpeg` (required by MoviePy for video rendering) and `Ollama` (local LLM server).

In [ ]:
# Install ffmpeg (required for MoviePy video rendering)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ ffmpeg installed")

# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh
print("✅ Ollama installed")

## 🚀 Step 3 — Start Ollama & Pull the LLM Model

Start the Ollama server in the background and download the `gemma3:4b` model.

> ⏳ **First run**: Model download takes ~3–5 minutes depending on connection speed.

In [ ]:
import subprocess
import time

# Start Ollama server in background
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)  # Wait for server to start
print("✅ Ollama server started (PID: {})".format(process.pid))

# Pull the Gemma 3 model
print("\n📥 Downloading gemma3:4b model (this may take a few minutes)...")
!ollama pull gemma3:4b
print("\n✅ Model ready!")

## ⚙️ Step 4 — Configure Environment

Set the pipeline configuration. **No API tokens needed** — image generation runs locally on GPU using Stable Diffusion XL!

In [ ]:
import os
import torch

# ════════════════════════════════════════════════════════════
# ⚙️  Pipeline Configuration
# ════════════════════════════════════════════════════════════

# --- LLM ---
os.environ["OLLAMA_URL"] = "http://localhost:11434/api/generate"
os.environ["OLLAMA_MODEL"] = "gemma3:4b"

# --- Image Generation (Stable Diffusion XL — runs locally on GPU) ---
os.environ["SD_MODEL_ID"] = "stabilityai/stable-diffusion-xl-base-1.0"
os.environ["SD_NUM_STEPS"] = "30"       # 20-50 (higher = better quality, slower)
os.environ["SD_GUIDANCE_SCALE"] = "7.5"  # 5-15 (higher = more prompt-adherent)

# --- Audio ---
os.environ["KOKORO_VOICE"] = "af_heart"

# --- Transcription ---
os.environ["WHISPER_MODEL"] = "medium.en"  # Options: tiny, base, small, medium, large
os.environ["WHISPER_CACHE_DIR"] = "/content/whisper_cache"

# --- Video ---
os.environ["VIDEO_FPS"] = "30"
os.environ["VIDEO_BITRATE"] = "20000k"
os.environ["MAX_SHORTS_DURATION"] = "60"

# --- Paths ---
os.environ["OUTPUT_DIR"] = "output"
os.environ["ASSETS_DIR"] = "assets"
os.environ["FONT_PATH"] = "assets/fonts/Anton-Regular.ttf"
os.environ["BACKGROUND_MUSIC"] = "assets/audio/background_music.mp3"
os.environ["LOG_FILE"] = "logs/colab.log"
os.environ["LOG_LEVEL"] = "INFO"

# GPU check
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else "None"
gpu_mem = f"{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if gpu_available else "N/A"

print("✅ Configuration set!")
print(f"\n🖥️  GPU: {gpu_name} ({gpu_mem})")
print(f"🤖 LLM: {os.environ['OLLAMA_MODEL']}")
print(f"🎨 Image: Stable Diffusion XL (local, {os.environ['SD_NUM_STEPS']} steps)")
print(f"🗣️  Voice: {os.environ['KOKORO_VOICE']}")
print(f"📝 Whisper: {os.environ['WHISPER_MODEL']}")

if not gpu_available:
    print("\n⚠️  WARNING: No GPU detected!")
    print("   Go to Runtime > Change runtime type > Select T4 GPU")
    print("   Image generation will be extremely slow without a GPU.")

## 📂 Step 5 — Setup Assets

Create required directories and check for font/music files.

Upload your **font file** and **background music** using the file browser on the left.

In [ ]:
from pathlib import Path

# Create asset directories
for d in ["assets/fonts", "assets/audio", "output", "logs", "config"]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"📁 Created: {d}/")

# Check for required assets
font_path = Path(os.environ["FONT_PATH"])
music_path = Path(os.environ["BACKGROUND_MUSIC"])

print("\n" + "═" * 50)
print("Asset Status:")
print("═" * 50)
print(f"  Font:  {'✅ Found' if font_path.exists() else '❌ Missing — upload to ' + str(font_path)}")
print(f"  Music: {'✅ Found' if music_path.exists() else '⚠️  Missing (optional) — upload to ' + str(music_path)}")

if not font_path.exists():
    print("\n📎 Download Anton font from:")
    print("   https://fonts.google.com/specimen/Anton")
    print("   Then upload Anton-Regular.ttf to assets/fonts/")

## 🎯 Step 6 — Run the Full Pipeline

Enter your motivational quote below and run the cell to generate the video.

The pipeline will:
1. Generate a story from your prompt using Gemma3
2. Create images locally using Stable Diffusion XL on the T4 GPU
3. Generate speech audio, transcribe it, and render the final video

> ⏳ **Estimated time**: 8–15 minutes (first run downloads the SD model ~6.5GB)

In [ ]:
# ════════════════════════════════════════════════════════════
# ✏️ Enter your motivational quote / prompt here:
# ════════════════════════════════════════════════════════════
PROMPT = "The greatest glory in living lies not in never falling, but in rising every time we fall."

# Optional: scheduled publish time (ISO-8601 format)
SCHEDULED_TIME = None  # e.g., "2025-06-01T15:00:00+05:30"

# ════════════════════════════════════════════════════════════
# 🚀 Run the pipeline
# ════════════════════════════════════════════════════════════
from video_engine.core.logger import setup_logging
from video_engine.core.pipeline import Pipeline

setup_logging()
pipeline = Pipeline()
result = pipeline.run(prompt=PROMPT, scheduled_time=SCHEDULED_TIME)

# Print results
print("\n" + "═" * 60)
if result["success"]:
    print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
else:
    print(f"❌ PIPELINE FAILED: {result['error']}")
print("═" * 60)
print(f"Total time: {result['total_duration_s']:.1f}s")
print("\nStep breakdown:")
for step in result.get("steps", []):
    icon = "✅" if step["success"] else "❌"
    print(f"  {icon} {step['step']} — {step['duration_s']:.1f}s")

## ▶️ Step 7 — Preview Generated Videos

Play the generated videos directly in the notebook.

In [ ]:
from IPython.display import Video, display, HTML
from pathlib import Path

output_dir = Path("output")
videos = list(output_dir.rglob("*.mp4"))

if not videos:
    print("⚠️  No videos found in output/ — run the pipeline first.")
else:
    print(f"🎬 Found {len(videos)} video(s):\n")
    for v in videos:
        print(f"📹 {v}")
        try:
            display(Video(str(v), embed=True, width=480))
        except Exception:
            print(f"   (Preview not available — download to view)")
        print()

## 📥 Step 8 — Download Generated Files

Download the generated videos, story, and SEO content to your local machine.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

output_dir = Path("output")
zip_name = "generated_videos.zip"

all_files = list(output_dir.rglob("*"))
output_files = [f for f in all_files if f.is_file()]

if not output_files:
    print("⚠️  No output files found — run the pipeline first.")
else:
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in output_files:
            zf.write(f, f.relative_to(output_dir))
            print(f"  📎 Added: {f.relative_to(output_dir)}")

    print(f"\n📦 Created: {zip_name}")
    print("📥 Downloading...")
    files.download(zip_name)

---

## 🔄 Run Individual Steps (Optional)

If you want to run specific pipeline steps individually for debugging or customisation.

### 📝 Generate Story Only

In [ ]:
from video_engine.core.config import get_settings
from video_engine.generators.story import generate_story

settings = get_settings()
settings.ensure_directories()

story = generate_story(
    "The only way to do great work is to love what you do.",
    settings
)
print("\n" + "═" * 50)
print(story)
print("═" * 50)

### 🎨 Generate Image Only (Stable Diffusion XL)

In [ ]:
from video_engine.core.config import get_settings
from video_engine.generators.image import generate_images
from pathlib import Path
from IPython.display import display
from PIL import Image

settings = get_settings()
work_dir = settings.video_output_dir
work_dir.mkdir(parents=True, exist_ok=True)

# Make sure prompt.txt exists (generate a story first, or write your own)
prompt_path = work_dir / "prompt.txt"
if not prompt_path.exists():
    prompt_path.write_text(
        "A child standing on a rooftop gazing at a starlit sky, feeling hopeful about the future",
        encoding="utf-8"
    )

generate_images(work_dir, settings)

# Display generated images
bg_dir = Path(settings.OUTPUT_DIR) / "background_file"
for img_file in sorted(bg_dir.glob("*.jpg")):
    print(f"\n🖼️  {img_file.name}")
    display(Image.open(img_file))

### 🗣️ Generate Audio Only

In [ ]:
from video_engine.core.config import get_settings
from video_engine.generators.audio import generate_audio
from IPython.display import Audio, display

settings = get_settings()
work_dir = settings.video_output_dir

audio_path = generate_audio(work_dir, settings)
print(f"\n🔊 Audio generated: {audio_path}")
display(Audio(str(audio_path)))

### 🧹 Free GPU Memory

Run this cell to free GPU memory after image generation if you need more VRAM for other tasks.

In [ ]:
from video_engine.generators.image import unload_model
import torch, gc

unload_model()
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    total_mem = torch.cuda.mem_get_info()[1] / 1e9
    print(f"✅ GPU memory freed: {free_mem:.1f} / {total_mem:.1f} GB available")
else:
    print("✅ Memory cleaned (no GPU)")

---

## 📋 Troubleshooting

| Issue | Solution |
|-------|----------|
| `No GPU detected` | Go to `Runtime > Change runtime type > T4 GPU` |
| `CUDA out of memory` | Run the 🧹 Free GPU Memory cell above, or use fewer inference steps (`SD_NUM_STEPS=20`) |
| `Ollama request failed` | Re-run Step 3 to restart the Ollama server |
| `Font not found` | Upload `Anton-Regular.ttf` to `assets/fonts/` |
| `ffmpeg not found` | Re-run Step 2 |
| Video won't preview | Download the zip and play locally |
| Slow image generation | Reduce `SD_NUM_STEPS` to 20 in Step 4 |

---

**Repository**: [github.com/tamilarasu18/ai-youtube-automation](https://github.com/tamilarasu18/ai-youtube-automation)  
**License**: MIT